# 3.5. Concise Implementation of Linear Regression

Let's build our first machine learning model with MindSpore! Since this is our first model, let's start with something simple - a linear regression model.

We'll see how to define, train and evaluate our model using MindSpore. The synthetic regression data we generated with scikit-learn in the previous chapter will come in handy. But first, let's import and initialize MindSpore 2.8.0 with the Ascend 310B1 NPU chip on our OrangePi AIpro \(20T\) development board.

In [1]:
import mindspore
mindspore.set_device(device_target='Ascend', device_id=0)

/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:146: SyntaxWarning: invalid escape sequence '\c'
  2. In forward, tiling would not split c1 and c0, find c1\c0 based on t2.
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:172: SyntaxWarning: invalid escape sequence '\c'
  1. Forward: tiling would not split c1\c0\h0, find c1\c0\h1\h0 based on t2
/home/HwHiAiUser/.pyenv/versions/3.12.13/envs/orangepiaipro-20t/lib/python3.12/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/home/HwHiAiUser/.pyenv/versions/3.12.13/envs/orangep

## 3.5.1. Defining the Model

Let's define our neural network. Our neural network should consist of a single, fully connected layer. By this we mean all inputs are connected to each output via matrix-vector multiplication.

In MindSpore 2.8.0, we define our network by creating a Python class inheriting from `mindspore.nn.Cell` and define the `construct` method. Assuming the shape of the synthetic data we generated in the previous chapter, our single fully connected layer should have exactly 2 input channels and 1 output channel. The fully connected layer is given by the `mindspore.nn.Dense` class.

Let's have our model accept an optional parameter `lr` for the learning rate which defaults to `0.01`. For the purposes of this chapter, we'll train our model with a learning rate of `0.03`.

In [2]:
from mindspore import nn
from mindspore import dtype as mstype

class MyModel(nn.Cell):
    def __init__(self, lr=0.01):
        super().__init__()
        self.lr = lr
        self.dense1 = nn.Dense(2, 1, dtype=mstype.float16)

    def construct(self, x):
        y_hat = self.dense1(x)
        return y_hat

model = MyModel(lr=0.03)
model

MyModel(
  (dense1): Dense(input_channels=2, output_channels=1, has_bias=True)
)

Note that we specified `dtype=mstype.float16` in our fully connected layer. That's because the Ascend 310B1 firmware implements matrix-vector and matrix-matrix multiplication only for 16-bit floating point values. `float16` has lower precision than their 32-bit and 64-bit counterparts but should suffice for our purposes.

## 3.5.2. Defining the Loss Function

Let's use the mean squared error as our loss function. In MindSpore, this is given by the class `mindspore.nn.MSELoss`.

In [3]:
loss_fn = nn.MSELoss()
loss_fn

MSELoss()

## 3.5.3. Defining the Optimization Algorithm

Let's use minibatch SGD for our optimization algorithm. In MindSpore 2.8.0, this is provided by the `mindspore.nn.SGD` class. Its constructor accepts 2 arguments.

1. The parameters to optimize over. This is given by the `trainable_params` method on our neural network
1. The learning rate required by our optimization algorithm, provided through the `learning_rate` argument

In [4]:
optimizer = nn.SGD(model.trainable_params(), model.lr)
optimizer

SGD()

## 3.5.4. Training

With our neural network, loss function and optimizer in place, let's generate some synthetic data as in the previous chapter and train our model on the data.

Define our forward function as below.

In [5]:
def forward_fn(X, y):
    y_hat = model(X)
    loss = loss_fn(y_hat, y)
    return loss, y_hat

Get the gradient function.

In [6]:
grad_fn = mindspore.value_and_grad(forward_fn, None, optimizer.parameters, has_aux=True)

Define our training step.

In [7]:
def train_step(X, y):
    (loss, _), grads = grad_fn(X, y)
    optimizer(grads)
    return loss

Now define our training function on the entire dataset.

In [8]:
def train(model, dataset):
    size = dataset.get_dataset_size()
    model.set_train()
    for batch_idx, (X_batch, y_batch) in enumerate(dataset.create_tuple_iterator()):
        loss = train_step(X_batch, y_batch)

        if batch_idx % 5 == 0:
            print(f'Training loss (scaled): {loss.asnumpy():>7f} [{batch_idx:>3d}/{size:>3d}]')

Generate the synthetic regression data as from our previous chapter and split them into training and testing datasets. Due to the `float16` precision limitation of matrix-vector and matrix-matrix multiplication imposed by the Ascend310B1 firmware, we must cast the synthetic features and labels returned by `make_regression` from the default `np.float64` to `np.float16`.

In [9]:
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
import numpy as np

bias = 4.2
X, y, coef = make_regression(n_samples=1000,
                             n_features=2,
                             n_targets=1,
                             bias=bias,
                             noise=0.01,
                             coef=True)
X, y = X.astype(np.float16), y.astype(np.float16).reshape(-1, 1)
X_train, X_test, y_train, y_test = train_test_split(X, y)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((750, 2), (250, 2), (750, 1), (250, 1))

Before training our model on the dataset, notice that our labels `y` are spread out with a non-zero mean and large value for the variance \(or standard deviation\). Our linear regression model performs best when both the features and labels approximate a standard normal distribution, i.e. bell-shaped with $\mu = 0$ and $\sigma^{2} = 1$.

`make_regression` produces well conditioned features by default, i.e. $\mu = 0$ and $\sigma^{2} = 1$. However, the labels often do not follow the same distribution. To solve this problem, we introduce the [`sklearn.preprocessing.StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) class which provides the folowing methods.

1. `fit_transform`: Computes the mean and standard deviation of the dataset and transforms the dataset to have mean $\mu = 0$ and standard deviation $\sigma = 1$
1. `inverse_transform`: Based on the mean and standard deviation computed in `fit_transform`, applies the inverse transformation to the scaled dataset, recovering their original values

In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
print(f'Before scaling: mean = {y_train.mean():.4f}, stddev = {y_train.std():.4f}')
y_train_scaled = scaler.fit_transform(y_train)
print(f'After scaling: mean = {y_train_scaled.mean():.4f}, stddev = {y_train_scaled.std():.4f}')

Before scaling: mean = 4.8984, stddev = inf
After scaling: mean = 0.0001, stddev = 1.0000


/home/HwHiAiUser/.pyenv/versions/3.12.13/envs/orangepiaipro-20t/lib/python3.12/site-packages/numpy/core/_methods.py:176: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)


Convert the training dataset into `mindspore.dataset.NumpySlicesDataset` and split into batches of size `32`, then use it to train our model.

In [11]:
import mindspore.dataset as ds

train_ds = ds.NumpySlicesDataset(data=(X_train, y_train_scaled), column_names=['features', 'labels'], shuffle=True)
train_ds = train_ds.batch(batch_size=32)
max_epochs = 3
for i in range(max_epochs):
    print(f'Epoch {i} start')
    train(model, train_ds)
    print(f'Epoch {i} end')

Epoch 0 start


/usr/local/Ascend/cann-8.5.0/python/site-packages/asc_op_compile_base/asc_op_compiler/ascendc_compile_gen_code.py:161: SyntaxWarning: invalid escape sequence '\w'
  match = re.search(f'{option}=(\w+)', ' '.join(compile_options))
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/c

.Training loss (scaled): 2.806641 [  0/ 24]
Training loss (scaled): 1.620117 [  5/ 24]
Training loss (scaled): 1.048828 [ 10/ 24]
Training loss (scaled): 0.539551 [ 15/ 24]
Training loss (scaled): 0.395508 [ 20/ 24]
Epoch 0 end
Epoch 1 start
Training loss (scaled): 0.103333 [  0/ 24]
Training loss (scaled): 0.088928 [  5/ 24]
Training loss (scaled): 0.044647 [ 10/ 24]
Training loss (scaled): 0.029449 [ 15/ 24]
Training loss (scaled): 0.010818 [ 20/ 24]
Epoch 1 end
Epoch 2 start
Training loss (scaled): 0.005043 [  0/ 24]
Training loss (scaled): 0.005611 [  5/ 24]
Training loss (scaled): 0.002874 [ 10/ 24]
Training loss (scaled): 0.001993 [ 15/ 24]
Training loss (scaled): 0.001408 [ 20/ 24]
Epoch 2 end


Finally, let's validate our model against our evaluation \(testing\) dataset. Notice the loss is negligible considering the variance of the given labels prior to scaling. This is because linear regression has a closed-form solution which is not true in general for machine learning problems.

Despite most machine learning problems lacking a closed-form solution, a sufficiently optimized algorithm with a carefully crafted neural network, loss and optimizer functions along with a suitable learning rate and batch size should produce models that provide highly accurate prediction for the given problem domain.

In [12]:
model.set_train(False)
y_hat_scaled = model(mindspore.Tensor(X_test)).asnumpy()
y_test_scaled = scaler.transform(y_test)
evaluation_loss_scaled = loss_fn(mindspore.Tensor(y_hat_scaled), mindspore.Tensor(y_test_scaled))
print(f'Evaluation loss (scaled): {evaluation_loss_scaled.asnumpy():>4f}')
y_hat = scaler.inverse_transform(y_hat_scaled)
evaluation_loss = loss_fn(mindspore.Tensor(y_hat), mindspore.Tensor(y_test))
print(f'Evaluation loss: {evaluation_loss.asnumpy():>4f}')

Evaluation loss (scaled): 0.000563
Evaluation loss: 4.062500


## 3.5.5. Summary

We saw in this chapter how to train our first linear regression model with MindSpore 2.8.0. While linear regression models are simple, almost trivial, they form the building blocks of deeper, more complex neural networks which we'll see in later chapters.